In [7]:
!pip install faker

import sqlite3
import csv
import zipfile
from faker import Faker
import random

f = Faker('fr_FR')

con = sqlite3.connect(':memory:')
cur = con.cursor()

ldd = """
CREATE TABLE ORGANISME(siret INT, nomOrganisme VARCHAR(50), raisonSociale TEXT, PRIMARY KEY(siret));
CREATE TABLE RANG(nomRang VARCHAR(50), PRIMARY KEY(nomRang));
CREATE TABLE TITRE(nomTitre VARCHAR(50), PRIMARY KEY(nomTitre));
CREATE TABLE DIGNITE(nomDignite VARCHAR(50), PRIMARY KEY(nomDignite));
CREATE TABLE GRADE(nomGrade VARCHAR(50), PRIMARY KEY(nomGrade));
CREATE TABLE GRADE_SUIVANT(nomGrade VARCHAR(50), nomGradSuiv VARCHAR(50), PRIMARY KEY(nomGrade, nomGradSuiv), UNIQUE(nomGrade), FOREIGN KEY(nomGrade) REFERENCES GRADE(nomGrade));
CREATE TABLE ORGANISATION(numOrganisation INT, nomOrganisation VARCHAR(50), PRIMARY KEY(numOrganisation));
CREATE TABLE TERRITOIRE(idTerritoire INT, nomTerritoire VARCHAR(50), PRIMARY KEY(idTerritoire));
CREATE TABLE ALIMENT(nomA VARCHAR(50), PRIMARY KEY(nomA));
CREATE TABLE LEGUME(nomLegume VARCHAR(50), PRIMARY KEY(nomLegume));
CREATE TABLE SAUCE(nomSauce VARCHAR(50), PRIMARY KEY(nomSauce));
CREATE TABLE MODELE(nomModele VARCHAR(50), typeEntretien VARCHAR(50), periodiciteEntretien TEXT, PRIMARY KEY(nomModele));
CREATE TABLE INGREDIENT(nomIngredient VARCHAR(50), PRIMARY KEY(nomIngredient));
CREATE TABLE TENRAC(codeTenrac INT, nomTenrac VARCHAR(50), courrielT VARCHAR(50), telT VARCHAR(50), adresseT VARCHAR(50), numOrganisation INT NOT NULL, nomGrade VARCHAR(50) NOT NULL, nomRang VARCHAR(50), nomTitre VARCHAR(50), nomDignite VARCHAR(50), siret INT NOT NULL, PRIMARY KEY(codeTenrac), FOREIGN KEY(numOrganisation) REFERENCES ORGANISATION(numOrganisation), FOREIGN KEY(nomGrade) REFERENCES GRADE(nomGrade), FOREIGN KEY(nomRang) REFERENCES RANG(nomRang), FOREIGN KEY(nomTitre) REFERENCES TITRE(nomTitre), FOREIGN KEY(nomDignite) REFERENCES DIGNITE(nomDignite), FOREIGN KEY(siret) REFERENCES ORGANISME(siret));
CREATE TABLE ORDRE(numOrganisation INT, idTerritoire INT NOT NULL, PRIMARY KEY(numOrganisation), UNIQUE(idTerritoire), FOREIGN KEY(numOrganisation) REFERENCES ORGANISATION(numOrganisation), FOREIGN KEY(idTerritoire) REFERENCES TERRITOIRE(idTerritoire));
CREATE TABLE LIEU(idLieu INT, adresseLieu VARCHAR(50), numOrganisation INT NOT NULL, PRIMARY KEY(idLieu), FOREIGN KEY(numOrganisation) REFERENCES ORDRE(numOrganisation));
CREATE TABLE PLAT(idPlat INT, nomLegume VARCHAR(50), PRIMARY KEY(idPlat), FOREIGN KEY(nomLegume) REFERENCES LEGUME(nomLegume));
CREATE TABLE MACHINE(nomMachine VARCHAR(50), nomModele VARCHAR(50) NOT NULL, PRIMARY KEY(nomMachine), FOREIGN KEY(nomModele) REFERENCES MODELE(nomModele));
CREATE TABLE ENTRETIEN(idEntretien INT, date_ DATE, nomMachine VARCHAR(50) NOT NULL, numOrganisation INT NOT NULL, codeTenrac INT NOT NULL, PRIMARY KEY(idEntretien), FOREIGN KEY(nomMachine) REFERENCES MACHINE(nomMachine), FOREIGN KEY(numOrganisation) REFERENCES ORGANISATION(numOrganisation), FOREIGN KEY(codeTenrac) REFERENCES TENRAC(codeTenrac));
CREATE TABLE CLUB(numOrganisation_1 INT, numOrganisation INT, PRIMARY KEY(numOrganisation_1), FOREIGN KEY(numOrganisation_1) REFERENCES ORGANISATION(numOrganisation), FOREIGN KEY(numOrganisation) REFERENCES ORDRE(numOrganisation));
CREATE TABLE REUNION(idReunion INT, dateReunion DATE, intitule TEXT, nomMachine VARCHAR(50) NOT NULL, idLieu INT NOT NULL, PRIMARY KEY(idReunion), FOREIGN KEY(nomMachine) REFERENCES MACHINE(nomMachine), FOREIGN KEY(idLieu) REFERENCES LIEU(idLieu));
CREATE TABLE participe(codeTenrac INT, idReunion INT, PRIMARY KEY(codeTenrac, idReunion), FOREIGN KEY(codeTenrac) REFERENCES TENRAC(codeTenrac), FOREIGN KEY(idReunion) REFERENCES REUNION(idReunion));
CREATE TABLE est_composee(idReunion INT, idPlat INT, PRIMARY KEY(idReunion, idPlat), FOREIGN KEY(idReunion) REFERENCES REUNION(idReunion), FOREIGN KEY(idPlat) REFERENCES PLAT(idPlat));
CREATE TABLE constitue(idPlat INT, nomA VARCHAR(50), PRIMARY KEY(idPlat, nomA), FOREIGN KEY(idPlat) REFERENCES PLAT(idPlat), FOREIGN KEY(nomA) REFERENCES ALIMENT(nomA));
CREATE TABLE constitue2(idPlat INT, nomSauce VARCHAR(50), PRIMARY KEY(idPlat, nomSauce), FOREIGN KEY(idPlat) REFERENCES PLAT(idPlat), FOREIGN KEY(nomSauce) REFERENCES SAUCE(nomSauce));
CREATE TABLE compose(nomSauce VARCHAR(50), nomIngredient VARCHAR(50), PRIMARY KEY(nomSauce, nomIngredient), FOREIGN KEY(nomSauce) REFERENCES SAUCE(nomSauce), FOREIGN KEY(nomIngredient) REFERENCES INGREDIENT(nomIngredient));
"""
cur.executescript(ldd)

grades = ['Affilié', 'Sympathisant', 'Adhérent', 'Chevalier / Dame', 'Commandeur', 'Grand''Croix']
rangs = ['Novice', 'Compagnon']
titres = ['Philanthrope', 'Protecteur', 'Honorable']
dignites = ['Maître', 'Grand Chancelier', 'Grand Maître']

cur.executemany("INSERT INTO GRADE VALUES (?)", [(g,) for g in grades])
cur.executemany("INSERT INTO RANG VALUES (?)", [(r,) for r in rangs])
cur.executemany("INSERT INTO TITRE VALUES (?)", [(t,) for t in titres])
cur.executemany("INSERT INTO DIGNITE VALUES (?)", [(d,) for d in dignites])

sirets = [10000000000000 + i for i in range(1, 11)]
for s in sirets:
    cur.execute("INSERT INTO ORGANISME VALUES (?, ?, ?)", (s, f.company(), f.catch_phrase()))
for i in range(1, 6):
    cur.execute("INSERT INTO TERRITOIRE VALUES (?, ?)", (i, f.region()))

for i in range(1, 6):
    cur.execute("INSERT INTO ORGANISATION VALUES (?, ?)", (i, f"Ordre_{i}"))
    cur.execute("INSERT INTO ORDRE VALUES (?, ?)", (i, i))

for i in range(6, 21):
    cur.execute("INSERT INTO ORGANISATION VALUES (?, ?)", (i, f"Club_{i}"))
    cur.execute("INSERT INTO CLUB VALUES (?, ?)", (i, random.randint(1, 5)))

liste_tenracs = []
for i in range(1, 300001):
    rang = random.choice(rangs + [None, None])
    titre = random.choice(titres + [None, None, None])
    dignite = random.choice(dignites + [None, None, None, None])

    liste_tenracs.append((
        i,
        f.name(),
        f.email(),
        "0600000000",
        f.address().replace('\n', ' '),
        random.randint(1, 20),
        random.choice(grades),
        rang,
        titre,
        dignite,
        random.choice(sirets)
    ))

    if i % 100000 == 0:
        cur.executemany("INSERT INTO TENRAC VALUES (?,?,?,?,?,?,?,?,?,?,?)", liste_tenracs)
        liste_tenracs = []

with open('SAE_Base_Tenrac.sql', 'w', encoding='utf-8') as f_sql:
    for line in con.iterdump():
        f_sql.write('%s\n' % line)

with zipfile.ZipFile('SAE_SQL_Complet.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('SAE_Base_Tenrac.sql')

cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cur.fetchall()

fichiers_csv_crees = []

for table_name in tables:
    nom_table = table_name[0]

    if nom_table.startswith('sqlite_'):
        continue

    cur.execute(f"SELECT * FROM {nom_table}")

    nom_fichier_csv = f"{nom_table}.csv"
    with open(nom_fichier_csv, 'w', newline='', encoding='utf-8') as f_csv:
        writer = csv.writer(f_csv)

        if cur.description:
            en_tetes = [description[0] for description in cur.description]
            writer.writerow(en_tetes)
            writer.writerows(cur)

    fichiers_csv_crees.append(nom_fichier_csv)

with zipfile.ZipFile('SAE_CSV_Complet.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fichier in fichiers_csv_crees:
        zipf.write(fichier)

con.close()